Shashank's changes

In [1]:
import torch
from transformers import RTDetrForObjectDetection

# 1. Tải mô hình gốc từ Hugging Face
model = RTDetrForObjectDetection.from_pretrained('PekingU/rtdetr_r18vd')

# 2. Đóng gói theo cấu trúc dictionary giống hệt mô hình prune
ckpt = {
    'model': model.state_dict(),
    'config': model.config,
    'model_object': model
}

# 3. Lưu thành 1 file .pt duy nhất
torch.save(ckpt, 'rtdetr_r18vd.pt')


/home/huy/miniconda3/envs/env_cv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 526/526 [00:00<00:00, 8603.00it/s]


In [9]:
# =============================================================================
# Cell 1: Setup Environment, Mount Drive, Define Paths (MODIFIED)
# =============================================================================
import os
import sys
import torch
import gc
import copy
import glob
import random
import json
from collections import defaultdict, Counter
import traceback
import time # <--- Added for inference time measurement

print("--- Environment Setup ---")

# Set CUDA Launch Blocking (Optional but Recommended for Debugging GPU errors)
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
print("CUDA_LAUNCH_BLOCKING set to 1.")

# Check if in Colab
IN_COLAB = 'google.colab' in sys.modules

# Install necessary libraries
print("Installing required libraries...")
# Note: Installing tqdm separately as sometimes the notebook version conflicts
!pip install -q --upgrade transformers datasets accelerate evaluate timm Pillow safetensors pycocotools thop torch-pruning tqdm
print("Libraries installation attempt finished.")

# Import key libraries (do this after install)
try:
    import torch
    import torch.nn as nn
    import numpy as np
    from transformers import (
        AutoImageProcessor, AutoModelForObjectDetection, AutoConfig
    )
    import torchvision
    from tqdm.notebook import tqdm as tqdm_notebook # For notebook progress bars
    from tqdm import tqdm as tqdm_cli # For regular loops if needed
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader
    import torch_pruning as tp
    from thop import profile
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    print("Core libraries imported successfully.")
except ImportError as e:
    print(f"ERROR: Failed to import libraries: {e}")
    print("Please check the pip install logs above.")
    raise e

# Mount Google Drive if in Colab
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        try:
            drive.mount('/content/drive')
            print("Google Drive mounted.")
        except Exception as e_mount:
            print(f"Error mounting drive: {e_mount}")
            raise e_mount
    else:
        print("Google Drive already mounted.")
    base_drive_path = "/content/drive/MyDrive/"
else:
    base_drive_path = "./" # Adjust if running locally

# --- Configuration (MODIFIED) ---
print("\n--- Configuration ---")
# !!! Important: Adjust these paths to your actual Drive locations !!!
model_dir = os.path.join(base_drive_path, "/content/drive/MyDrive/deformable-detr-finetuned-kitti-round2") # DIR where fine-tuned model was saved
dataset_base_dir = os.path.join(base_drive_path, "kitti_subset") # Base DIR for KITTI subset
images_dir = os.path.join(dataset_base_dir, "images") # Specific image folder
# --- Make sure this points to your COCO format VALIDATION json ---
coco_annotation_file = os.path.join(dataset_base_dir, "annotations", "instances_val2017.json") # <== COCO format annotations for mAP
# --- Output directory ---
output_dir = os.path.join(base_drive_path, "kitti_torch_pruning_results_v2") # Output directory for this run

# Pruning Params
TARGET_PRUNING_RATIOS = [0.1, 0.2, 0.3, 0.4, 0.5] # Ratios to test (relative to original)

# Dataset Params (ensure these match your KITTI subset)
NUM_KITTI_CLASSES = 3 # Car, Pedestrian, Cyclist
NUM_OUTPUTS_REQUIRED = NUM_KITTI_CLASSES  # Add 1 for the background/no-object class

# Evaluation params
EVAL_BATCH_SIZE = 1 # Adjust based on GPU memory for evaluation inference speed
CONFIDENCE_THRESHOLD = 0.1 # Threshold for keeping detections during post-processing

# --- End Configuration ---

# Create output directory
os.makedirs(output_dir, exist_ok=True)
print(f"Model directory: {model_dir}")
print(f"Dataset directory: {dataset_base_dir}")
print(f"COCO Annotation file: {coco_annotation_file}")
print(f"Output directory: {output_dir}")
print(f"Target Pruning Ratios: {TARGET_PRUNING_RATIOS}")

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if not torch.cuda.is_available():
    print("WARNING: CUDA not available, running on CPU. This will be very slow.")

# Helper Functions (from original Cell 1)
def get_module_by_name(model: nn.Module, name: str) -> nn.Module:
    names = name.split('.')
    obj = model
    for n in names:
        if hasattr(obj, n): obj = getattr(obj, n)
        else:
            try: idx = int(n); obj = obj[idx]
            except (ValueError, IndexError, TypeError): raise AttributeError(f"Module part '{n}' not found in name '{name}'. Parent type: {type(obj)}")
    return obj

def set_module_by_name(model: nn.Module, name: str, new_module: nn.Module):
    names = name.split('.')
    parent_name = '.'.join(names[:-1])
    leaf_name = names[-1]
    try: parent_module = model.get_submodule(parent_name) if parent_name else model
    except AttributeError: parent_module = get_module_by_name(model, parent_name) # Fallback
    if hasattr(parent_module, leaf_name): setattr(parent_module, leaf_name, new_module)
    else:
        try: idx = int(leaf_name); parent_module[idx] = new_module
        except (ValueError, IndexError, TypeError): raise AttributeError(f"Could not set attribute or index '{leaf_name}' in parent module '{parent_name}' of type {type(parent_module)}")

print("\nSetup Complete.")

--- Environment Setup ---
CUDA_LAUNCH_BLOCKING set to 1.
Installing required libraries...
Libraries installation attempt finished.
Core libraries imported successfully.
Google Drive already mounted.

--- Configuration ---
Model directory: /content/drive/MyDrive/deformable-detr-finetuned-kitti-round2
Dataset directory: /content/drive/MyDrive/kitti_subset
COCO Annotation file: /content/drive/MyDrive/kitti_subset/annotations/instances_val2017.json
Output directory: /content/drive/MyDrive/kitti_torch_pruning_results_v2
Target Pruning Ratios: [0.1, 0.2, 0.3, 0.4, 0.5]
Using device: cuda

Setup Complete.


In [10]:
# =============================================================================
# Cell 2: Load & Prepare Base Model (Revised for 3 Classes & Correct Loading)
# =============================================================================
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModelForObjectDetection, AutoImageProcessor
import traceback
import gc

# --- Import Helper Functions from Cell 1 ---
assert 'set_module_by_name' in globals(), "Helper function set_module_by_name not defined (run Cell 1)"
assert 'NUM_OUTPUTS_REQUIRED' in locals() and NUM_OUTPUTS_REQUIRED == 3, "NUM_OUTPUTS_REQUIRED should be set to 3 in Cell 1"
assert 'model_dir' in locals(), "model_dir not defined in Cell 1"
assert 'device' in locals(), "device not defined in Cell 1"

print("\n--- Loading & Preparing Base Model (Targeting 3 Classes) ---")

prepared_base_model = None
image_processor = None
config = None
# Default values, might be overridden by loaded config
hidden_dim = 256
decoder_layers = 6
num_queries = 300

# --- Function to Replace FrozenBN (Essential for pruning/fine-tuning later) ---
def replace_frozen_bn(model_to_modify):
    FROZEN_BN_TYPE = None
    try:
        # Try importing the specific FrozenBN class
        from transformers.models.deformable_detr.modeling_deformable_detr import DeformableDetrFrozenBatchNorm2d
        FROZEN_BN_TYPE = DeformableDetrFrozenBatchNorm2d
        print("  Found DeformableDetrFrozenBatchNorm2d class.")
    except ImportError:
        print("  WARNING: DeformableDetrFrozenBatchNorm2d class not found. Cannot replace BN layers.")
        return model_to_modify, 0 # Return unmodified model if class not found

    if not FROZEN_BN_TYPE:
        return model_to_modify, 0

    replacement_count = 0
    error_count = 0
    module_list = list(model_to_modify.named_modules())
    modules_to_replace = []
    print(f"  Checking {len(module_list)} modules for FrozenBN replacement...")

    for name, module in module_list:
        if isinstance(module, FROZEN_BN_TYPE):
            modules_to_replace.append(name)

    if not modules_to_replace:
        print("  No FrozenBN layers found to replace.")
        return model_to_modify, 0

    print(f"  Found {len(modules_to_replace)} FrozenBN layers to replace.")
    # Perform replacement on CPU for safety, then move back
    original_device = next(iter(model_to_modify.parameters()), torch.tensor(0)).device # Get device robustly
    model_to_modify.cpu()
    print(f"    Moved model to CPU for BN replacement.")

    from tqdm import tqdm as tqdm_cli # Use standard tqdm for this internal loop
    for name in tqdm_cli(modules_to_replace, desc="  Replacing FrozenBN", leave=False):
        try:
            module = model_to_modify.get_submodule(name) # Use get_submodule
            if not isinstance(module, FROZEN_BN_TYPE): continue # Should not happen now

            # Check if weight exists to determine num_features
            if hasattr(module, 'weight') and module.weight is not None:
                num_features = module.weight.shape[0]
            else:
                # This might happen if the layer was somehow incomplete
                print(f"    WARNING: Skipping {name} - no weight attribute found to determine num_features.")
                error_count += 1
                continue

            # Create standard BN layer on CPU
            new_bn = nn.BatchNorm2d(num_features, eps=1e-5, affine=True, track_running_stats=True) # Created on CPU

            # Copy parameters from FrozenBN to standard BN *before* replacing
            if hasattr(module, 'weight') and module.weight is not None:
                 new_bn.weight.data.copy_(module.weight.data)
            if hasattr(module, 'bias') and module.bias is not None:
                 new_bn.bias.data.copy_(module.bias.data)
            if hasattr(module, 'running_mean') and module.running_mean is not None:
                 new_bn.running_mean.data.copy_(module.running_mean.data)
            if hasattr(module, 'running_var') and module.running_var is not None:
                 new_bn.running_var.data.copy_(module.running_var.data)
            # num_batches_tracked might not always exist or be relevant, copy if present
            if hasattr(module, 'num_batches_tracked') and module.num_batches_tracked is not None and hasattr(new_bn, 'num_batches_tracked'):
                 new_bn.num_batches_tracked.data.copy_(module.num_batches_tracked.data)

            # Replace the module using the helper function
            set_module_by_name(model_to_modify, name, new_bn)
            replacement_count += 1
        except Exception as e_replace:
            print(f"    ERROR replacing {name}: {e_replace}")
            traceback.print_exc()
            error_count += 1

    # Move back to original device
    model_to_modify.to(original_device)
    print(f"    Moved model back to {original_device}.")
    print(f"  Finished BN replacement. Replaced: {replacement_count}, Errors: {error_count}")
    if error_count > 0:
        print("  WARNING: Errors occurred during BN replacement. This might affect performance.")
    return model_to_modify, replacement_count
# --- End Replace Function ---

try:
    # 1. Load processor
    image_processor = AutoImageProcessor.from_pretrained(model_dir)
    print(f"Image processor loaded from {model_dir}")

    # 2. Load config and IMMEDIATELY ensure it has the correct number of labels (3)
    print(f"Loading config from {model_dir} and ensuring num_labels is {NUM_OUTPUTS_REQUIRED}...")
    # Define the correct 3-class label mapping expected by the KITTI fine-tuned model
    # IMPORTANT: Verify this matches the classes and order used during YOUR fine-tuning
    id2label = {0: 'Car', 1: 'Pedestrian', 2: 'Cyclist'}
    label2id = {v: k for k, v in id2label.items()}

    config = AutoConfig.from_pretrained(
        model_dir,
        num_labels=NUM_OUTPUTS_REQUIRED, # Explicitly set to 3
        id2label=id2label,               # Set correct mapping
        label2id=label2id                # Set correct mapping
    )

    # Update other parameters if needed (usually loaded correctly from config.json)
    num_queries = getattr(config, 'num_queries', 300)
    hidden_dim = getattr(config, 'd_model', 256)
    decoder_layers = getattr(config, 'decoder_layers', 6)
    print(f"Using config: num_labels={config.num_labels}, id2label={config.id2label}")
    print(f"              num_queries={num_queries}, hidden_dim={hidden_dim}, decoder_layers={decoder_layers}")


    # 3. Load model structure and weights using the CORRECTED config
    #    Use ignore_mismatched_sizes=False first. It should work now.
    print(f"Loading model weights from {model_dir}...")
    _model = AutoModelForObjectDetection.from_pretrained(
        model_dir,
        config=config,                  # Pass the corrected config
        ignore_mismatched_sizes=False   # <<< TRY THIS FIRST! Should match now.
    )
    print("Model weights loaded.")
     # If the above fails with size mismatch, something is still wrong with the checkpoint file
     # or the assumed id2label mapping. Only use ignore_mismatched_sizes=True as a last resort
     # and investigate why the checkpoint weights don't match the 3-label config.

    # 4. Verify Head Size (Optional but recommended sanity check)
    print("Verifying loaded model head size...")
    try:
        final_class_layer = None
        # Add robust checks to find the last linear layer in the classification head
        if hasattr(_model, 'class_embed'):
            if isinstance(_model.class_embed, nn.ModuleList) and len(_model.class_embed) > 0:
                 last_mod_in_list = _model.class_embed[-1]
                 if isinstance(last_mod_in_list, nn.Linear): final_class_layer = last_mod_in_list
                 elif hasattr(last_mod_in_list, 'layers') and isinstance(last_mod_in_list.layers, nn.Sequential):
                     if len(last_mod_in_list.layers) > 0 and isinstance(last_mod_in_list.layers[-1], nn.Linear): final_class_layer = last_mod_in_list.layers[-1]
            elif isinstance(_model.class_embed, nn.Linear): final_class_layer = _model.class_embed
            # Add more checks if your model structure is different

        if final_class_layer is not None:
            current_cls_outputs = final_class_layer.out_features
            print(f"  Detected {current_cls_outputs} outputs in the final classification layer.")
            if current_cls_outputs != NUM_OUTPUTS_REQUIRED:
                 # This should NOT happen if ignore_mismatched_sizes=False worked
                 print(f"  ERROR: Loaded model head size ({current_cls_outputs}) does not match required ({NUM_OUTPUTS_REQUIRED})!")
                 raise ValueError("Head size mismatch after loading with corrected config.")
            else:
                 print(f"  Head size matches required size ({NUM_OUTPUTS_REQUIRED}). Fine-tuned weights presumed loaded.")
        else:
            print("  WARNING: Could not reliably determine output size of the classification head.")

    except Exception as e_head_check:
        print(f"  Error during head size verification: {e_head_check}")
        raise e_head_check

    # 5. Replace FrozenBatchNorm2d layers (necessary for pruning)
    print("\nReplacing FrozenBatchNorm2d layers with standard BatchNorm2d...")
    _model, _ = replace_frozen_bn(_model) # Use the function defined at the start of this cell

    # 6. Move final prepared model to target device
    print(f"\nMoving final prepared base model to: {device}")
    _model.to(device)
    _model.eval()
    prepared_base_model = _model # Assign to the final variable name
    print("Prepared base model is ready on device.")

    # 7. Calculate and print parameters for the prepared model
    base_model_params = sum(p.numel() for p in prepared_base_model.parameters() if p.requires_grad)
    print(f"Prepared Base Model Parameters (trainable): {base_model_params:,}")

except Exception as e_load_prep:
    print(f"!!! ERROR during model loading/preparation: {e_load_prep} !!!")
    traceback.print_exc()
    prepared_base_model = None # Ensure it's None on error
    raise e_load_prep # Re-raise the exception to stop execution

# Cleanup intermediate model variable if it exists and differs
if '_model' in locals() and prepared_base_model is not _model:
    del _model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

print("\n--- Base Model Preparation Complete ---")


--- Loading & Preparing Base Model (Targeting 3 Classes) ---
Image processor loaded from /content/drive/MyDrive/deformable-detr-finetuned-kitti-round2
Loading config from /content/drive/MyDrive/deformable-detr-finetuned-kitti-round2 and ensuring num_labels is 3...
Using config: num_labels=3, id2label={0: 'Car', 1: 'Pedestrian', 2: 'Cyclist'}
              num_queries=300, hidden_dim=256, decoder_layers=6
Loading model weights from /content/drive/MyDrive/deformable-detr-finetuned-kitti-round2...
Model weights loaded.
Verifying loaded model head size...
  Detected 3 outputs in the final classification layer.
  Head size matches required size (3). Fine-tuned weights presumed loaded.

Replacing FrozenBatchNorm2d layers with standard BatchNorm2d...
  Found DeformableDetrFrozenBatchNorm2d class.
  Checking 425 modules for FrozenBN replacement...
  Found 53 FrozenBN layers to replace.
    Moved model to CPU for BN replacement.


    Moved model back to cpu.
  Finished BN replacement. Replaced: 53, Errors: 0

Moving final prepared base model to: cuda


Prepared base model is ready on device.
Prepared Base Model Parameters (trainable): 39,877,769

--- Base Model Preparation Complete ---


In [11]:
# =============================================================================
# Cell 3: Load COCO GT & Define Eval Dataset/Loader (MODIFIED)
# =============================================================================
print("\n--- Loading COCO GT and Preparing Evaluation Dataloader ---")

coco_gt = None
eval_loader = None

# --- Load COCO GT data ---
if not os.path.exists(coco_annotation_file):
    print(f"ERROR: COCO annotation file for evaluation not found at: {coco_annotation_file}")
    print("Cannot calculate mAP.")
else:
    try:
        print(f"Loading COCO ground truth for mAP evaluation from: {coco_annotation_file}")
        coco_gt = COCO(coco_annotation_file)
        print("COCO GT loaded successfully.")
    except Exception as e_coco:
        print(f"ERROR loading COCO annotations file '{coco_annotation_file}': {e_coco}")
        traceback.print_exc()
        coco_gt = None

# --- Define CocoEvalDataset ---
class CocoEvalDataset(Dataset):
    def __init__(self, coco_gt_obj, img_dir):
        self.coco = coco_gt_obj
        self.img_ids = coco_gt_obj.getImgIds()
        self.img_dir = img_dir
        self.img_info = coco_gt_obj.loadImgs(self.img_ids)
        # Create a mapping from image_id to file path
        self.id_to_path = {info['id']: os.path.join(img_dir, info['file_name'])
                           for info in self.img_info if 'file_name' in info}
        print(f"  CocoEvalDataset: Mapped {len(self.id_to_path)} image IDs to file paths.")
        # Verify paths exist
        missing_files = 0
        valid_img_ids = []
        for img_id in self.img_ids:
            path = self.id_to_path.get(img_id)
            if path and os.path.exists(path):
                valid_img_ids.append(img_id)
            else:
                missing_files += 1
        if missing_files > 0:
            print(f"  WARNING: Could not find image files for {missing_files} image IDs.")
        self.img_ids = valid_img_ids # Use only IDs with existing images
        print(f"  Using {len(self.img_ids)} valid image IDs for evaluation.")

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_path = self.id_to_path[img_id] # Should exist based on init check
        try:
            image = Image.open(img_path).convert("RGB")
            # Target dict needed for post-processing size info
            target = {"image_id": img_id, "width": image.width, "height": image.height}
            return image, target
        except Exception as e:
            print(f"Error loading/processing image {img_path} for id {img_id}: {e}")
            return None # Return None if image loading fails

# --- Create Evaluation DataLoader if coco_gt loaded ---
if coco_gt:
    # Ensure image_processor is available
    assert 'image_processor' in locals() and image_processor is not None, "Image processor not loaded in Cell 2."

    eval_dataset = CocoEvalDataset(coco_gt_obj=coco_gt, img_dir=images_dir)

    # Define a collate function for evaluation (handles None)
    def eval_collate_fn(batch):
        batch = [item for item in batch if item is not None]
        if not batch: return None
        # Collate images and targets separately
        images = [item[0] for item in batch]
        targets = [item[1] for item in batch]
        # Use image_processor to pad images
        try:
            # Note: return_tensors="pt" happens inside the loop when calling processor
            # Here we just need the list of images and targets
             return images, targets
        except Exception as e_pad:
            print(f"Error during custom collate: {e_pad}")
            return None

    eval_loader = DataLoader(eval_dataset,
                             batch_size=EVAL_BATCH_SIZE, # Use config batch size
                             shuffle=False,
                             num_workers=2, # Can increase if not causing issues
                             collate_fn=eval_collate_fn,
                             pin_memory=True if device.type == 'cuda' else False)
    print(f"Created evaluation DataLoader with {len(eval_dataset)} images and batch size {EVAL_BATCH_SIZE}.")
else:
    print("Evaluation DataLoader not created as COCO GT is missing.")

gc.collect()


--- Loading COCO GT and Preparing Evaluation Dataloader ---
Loading COCO ground truth for mAP evaluation from: /content/drive/MyDrive/kitti_subset/annotations/instances_val2017.json
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
COCO GT loaded successfully.
  CocoEvalDataset: Mapped 200 image IDs to file paths.
  Using 200 valid image IDs for evaluation.
Created evaluation DataLoader with 200 images and batch size 1.


0

In [12]:
# =============================================================================
# Cell 4: Calculate Baseline Metrics (MODIFIED)
# =============================================================================
print("\n--- Calculating Baseline Metrics ---")

# Ensure base model is available
assert 'prepared_base_model' in locals() and prepared_base_model is not None, "Run Cell 2 first"
assert 'image_processor' in locals() and image_processor is not None, "Run Cell 2 first"

baseline_results = {}

# 1. Parameters (already calculated in Cell 2)
base_model_params = sum(p.numel() for p in prepared_base_model.parameters() if p.requires_grad)
baseline_results['params'] = base_model_params
print(f"Baseline Parameters: {base_model_params:,}")

# 2. GFLOPs
print("Calculating baseline GFLOPs...")
base_model_gflops = -1.0
dummy_input = None
try:
    # Determine input size (consistent logic)
    bs = 1; img_h, img_w = 800, 1333 # Default/common size
    if hasattr(image_processor, 'size') and isinstance(image_processor.size, dict):
         size_dict = image_processor.size
         if 'shortest_edge' in size_dict:
             shortest = size_dict['shortest_edge']; max_size = getattr(image_processor, 'max_size', 1333); aspect_ratio = 1333 / 800
             if shortest == 800 and max_size == 1333: img_h, img_w = 800, 1333
             else: img_h = shortest; img_w = int(shortest * aspect_ratio);
             if img_w > max_size: img_w = max_size; img_h = int(max_size / aspect_ratio)
         elif 'height' in size_dict and 'width' in size_dict: img_h = size_dict['height']; img_w = size_dict['width']
         img_h = max(img_h, 32); img_w = max(img_w, 32) # Ensure min size
    print(f"  Using dummy input size H={img_h}, W={img_w} for GFLOPs calculation")
    dummy_input = torch.randn(bs, 3, img_h, img_w, device=device)

    # Profile on GPU
    flops, params_thop = profile(prepared_base_model, inputs=(dummy_input,), verbose=False)
    base_model_gflops = flops / 1e9
    print(f"Baseline GFLOPs: {base_model_gflops:.2f}")
    baseline_results['gflops'] = base_model_gflops
except Exception as e_gflops:
    print(f"  Error calculating GFLOPs: {e_gflops}")
    baseline_results['gflops'] = -1.0
    # Keep dummy_input if created, might be needed later
    if 'dummy_input' not in locals(): dummy_input = None


# 3. COCO mAP and Inference Time
print("\nCalculating baseline mAP and Average Inference Time...")
baseline_results['mAP'] = -1.0
baseline_results['mAP50'] = -1.0
baseline_results['avg_inference_ms'] = -1.0

if coco_gt is None or eval_loader is None:
    print("  Skipping mAP/Inference Time calculation (COCO GT or Eval Loader missing).")
else:
    coco_results_baseline = []
    total_inference_time_ms = 0
    processed_images_count = 0
    inference_batches = 0

    prepared_base_model.eval() # Ensure eval mode
    with torch.no_grad():
        for batch_data in tqdm_notebook(eval_loader, desc="Baseline Evaluation"):
            if batch_data is None: continue
            images, targets = batch_data # Unpack images and targets

            try:
                # --- Batch Preprocessing ---
                inputs = image_processor(images=images, return_tensors="pt").to(device)
                original_sizes = [(t['height'], t['width']) for t in targets]
                target_sizes = torch.tensor(original_sizes, device=device)
                image_ids = [t['image_id'] for t in targets]

                # --- Timed Inference ---
                start_time = time.perf_counter()
                outputs = prepared_base_model(**inputs)
                end_time = time.perf_counter()
                total_inference_time_ms += (end_time - start_time) * 1000
                inference_batches += 1
                # --- End Timed Inference ---

                # --- Post-processing ---
                results_list = image_processor.post_process_object_detection(
                    outputs,
                    target_sizes=target_sizes,
                    threshold=CONFIDENCE_THRESHOLD
                )

                # --- Format for COCO ---
                for i, results in enumerate(results_list):
                    image_id = image_ids[i]
                    boxes = results["boxes"].cpu().tolist()
                    scores = results["scores"].cpu().tolist()
                    labels = results["labels"].cpu().tolist()
                    for box, score, label in zip(boxes, scores, labels):
                        x_min, y_min, x_max, y_max = box
                        coco_bbox = [x_min, y_min, x_max - x_min, y_max - y_min]
                        coco_results_baseline.append({
                            "image_id": image_id,
                            "category_id": label+1,
                            "bbox": coco_bbox,
                            "score": score,
                        })
                    processed_images_count += 1

            except Exception as e_infer:
                 print(f"\nError during baseline inference/postprocessing: {e_infer}")
                 traceback.print_exc()
                 continue # Skip batch on error

    print(f"\nBaseline: Processed {processed_images_count} images over {inference_batches} batches.")

    # Calculate Average Inference Time (per batch)
    if inference_batches > 0:
        baseline_results['avg_inference_ms'] = total_inference_time_ms / inference_batches
        print(f"Baseline Average Batch Inference Time: {baseline_results['avg_inference_ms']:.2f} ms")

    # Run COCOeval API
    if not coco_results_baseline:
        print("No baseline evaluation results generated.")
    else:
        print("Running COCO evaluation API for baseline...")
        try:
            coco_dt = coco_gt.loadRes(coco_results_baseline)
            coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
            coco_eval.evaluate(); coco_eval.accumulate(); coco_eval.summarize()
            baseline_results['mAP'] = coco_eval.stats[0]
            baseline_results['mAP50'] = coco_eval.stats[1]
            print(f"Baseline mAP: {baseline_results['mAP']:.4f}, mAP50: {baseline_results['mAP50']:.4f}")
            # Cleanup eval objects
            del coco_dt, coco_eval
        except Exception as e_coco_api:
            print(f"  ERROR during baseline COCOeval: {e_coco_api}")
            traceback.print_exc()

# Store dummy_input globally if needed for pruning step
if 'dummy_input' in locals() and dummy_input is not None:
     globals()['dummy_input'] = dummy_input
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("\nBaseline Metric Calculation Complete.")


--- Calculating Baseline Metrics ---
Baseline Parameters: 39,877,769
Calculating baseline GFLOPs...
  Using dummy input size H=800, W=1333 for GFLOPs calculation
Baseline GFLOPs: 204.87

Calculating baseline mAP and Average Inference Time...


Baseline Evaluation:   0%|          | 0/200 [00:00<?, ?it/s]


Baseline: Processed 200 images over 200 batches.
Baseline Average Batch Inference Time: 195.07 ms
Running COCO evaluation API for baseline...
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.48s).
Accumulating evaluation results...
DONE (t=0.08s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.117
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.264
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.095
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.066
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.128
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.178
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.088
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.270
 Aver

In [13]:
# prune_model.py (or paste into a notebook cell)
import copy
import random
import torch
import torch.nn as nn
import math
from tqdm import tqdm as tqdm_cli # Optional for progress if needed inside

def simplified_prune_detr(model, prune_ratio=0.2):
    """
    Prune Deformable DETR (Hugging Face version) by removing transformer layers only.
    Correctly handles re-indexing of auxiliary prediction heads based on
    the actual number of heads present in the original model. Ensures heads
    are assigned at the top-level model attribute.

    Args:
        model: Original Deformable DETR model instance (Hugging Face version,
               e.g., DeformableDetrForObjectDetection).
        prune_ratio: Percentage of layers to prune (0.0 to 1.0).

    Returns:
        Pruned model (a deep copy of the original, modified).
    """
    print(f"\n--- Starting Simplified Layer Pruning (Ratio: {prune_ratio:.2f}) ---")
    # Create a deep copy to avoid modifying the original model instance
    pruned_model = copy.deepcopy(model)
    pruned_model.eval() # Ensure model is in eval mode after copy

    # --- Structure Check for Hugging Face Model ---
    # Check for the nested 'model' attribute which contains the core components
    if not hasattr(pruned_model, 'model') or not isinstance(pruned_model.model, nn.Module):
         print("Error: Top-level model does not have a 'model' attribute. Cannot find transformer components.")
         return model # Return original if structure is wrong

    # Access the core DeformableDetrModel components
    nested_model = pruned_model.model
    if not hasattr(nested_model, 'encoder') or not isinstance(nested_model.encoder, nn.Module):
        print("Error: Nested 'model' attribute does not have an 'encoder' module.")
        return model
    if not hasattr(nested_model, 'decoder') or not isinstance(nested_model.decoder, nn.Module):
        print("Error: Nested 'model' attribute does not have a 'decoder' module.")
        return model
    # --- End Structure Check ---

    # Get encoder and decoder modules
    encoder = nested_model.encoder
    decoder = nested_model.decoder

    # Get original layer counts safely
    original_enc_layers = len(encoder.layers) if hasattr(encoder, 'layers') and isinstance(encoder.layers, nn.ModuleList) else 0
    original_dec_layers = len(decoder.layers) if hasattr(decoder, 'layers') and isinstance(decoder.layers, nn.ModuleList) else 0

    print(f"Original encoder layers: {original_enc_layers}")
    print(f"Original decoder layers: {original_dec_layers}")

    # --- Calculate Layers to Prune ---
    if original_enc_layers == 0 and original_dec_layers == 0:
        print("Model has no encoder or decoder layers. Skipping pruning.")
        # Ensure layer counts are set on the top-level pruned_model if needed elsewhere
        # These attributes might not exist on the original HF model, so add them if needed
        if not hasattr(pruned_model,'num_encoder_layers'): pruned_model.num_encoder_layers = 0
        if not hasattr(pruned_model,'num_decoder_layers'): pruned_model.num_decoder_layers = 0
        return pruned_model

    enc_layers_to_prune = math.ceil(original_enc_layers * prune_ratio) if original_enc_layers > 0 else 0
    dec_layers_to_prune = math.ceil(original_dec_layers * prune_ratio) if original_dec_layers > 0 else 0

    # Ensure we don't prune all layers if layers exist and pruning is attempted
    enc_layers_to_prune = min(enc_layers_to_prune, original_enc_layers - 1) if original_enc_layers > 1 else 0
    dec_layers_to_prune = min(dec_layers_to_prune, original_dec_layers - 1) if original_dec_layers > 1 else 0

    print(f"Targeting removal of {enc_layers_to_prune} encoder layers.")
    print(f"Targeting removal of {dec_layers_to_prune} decoder layers.")

    # --- Encoder Pruning ---
    new_encoder_layers = encoder.layers # Default to existing list
    if original_enc_layers > 0 and enc_layers_to_prune > 0:
        # Randomly sample indices to KEEP
        enc_indices_to_keep = sorted(random.sample(range(original_enc_layers),
                                                original_enc_layers - enc_layers_to_prune))
        print(f"Keeping encoder layers at original indices: {enc_indices_to_keep}")
        # Create a new ModuleList with only the kept layers
        new_encoder_layers = nn.ModuleList([encoder.layers[i] for i in enc_indices_to_keep])
        encoder.layers = new_encoder_layers # Modify encoder within nested_model
    else:
        print("Skipping encoder pruning (0 layers or 0 prune count).")

    # Store actual final encoder layer count on the TOP-LEVEL model object for potential later use
    # Create the attribute if it doesn't exist
    pruned_model.num_encoder_layers = len(new_encoder_layers)


    # --- Decoder Pruning ---
    new_decoder_layers = decoder.layers # Default to existing list
    dec_indices_to_keep = list(range(original_dec_layers)) # Default to all indices before pruning
    if original_dec_layers > 0 and dec_layers_to_prune > 0:
        # Randomly sample indices to KEEP
        dec_indices_to_keep = sorted(random.sample(range(original_dec_layers),
                                                original_dec_layers - dec_layers_to_prune))
        print(f"Keeping decoder layers at original indices: {dec_indices_to_keep}")
        # Create a new ModuleList with only the kept layers
        new_decoder_layers = nn.ModuleList([decoder.layers[i] for i in dec_indices_to_keep])
        decoder.layers = new_decoder_layers # Modify decoder within nested_model
    else:
        print("Skipping decoder pruning (0 layers or 0 prune count).")

    # Update decoder's internal layer count attribute if it exists
    if hasattr(decoder, 'num_layers'):
        decoder.num_layers = len(new_decoder_layers)
    # Store actual final decoder layer count on the TOP-LEVEL model object
    pruned_model.num_decoder_layers = len(new_decoder_layers)


    # --- Prediction Head Pruning and Re-indexing ---
    # Heads are usually attributes of the TOP-LEVEL 'pruned_model' object
    has_class_embed = hasattr(pruned_model, 'class_embed') and pruned_model.class_embed is not None
    has_bbox_embed = hasattr(pruned_model, 'bbox_embed') and pruned_model.bbox_embed is not None

    # Check if heads are ModuleLists and have multiple elements (indicating aux heads)
    is_multi_head_class = has_class_embed and isinstance(pruned_model.class_embed, nn.ModuleList) and len(pruned_model.class_embed) > 1
    is_multi_head_bbox = has_bbox_embed and isinstance(pruned_model.bbox_embed, nn.ModuleList) and len(pruned_model.bbox_embed) > 1

    # Rebuild head ModuleLists ONLY if the original model used auxiliary heads
    if is_multi_head_class or is_multi_head_bbox:
        print("\nRebuilding prediction heads ModuleList for pruned decoder...")

        # Initialize new lists (or None) based on original existence
        new_class_embed_list = nn.ModuleList() if has_class_embed else None
        new_bbox_embed_list = nn.ModuleList() if has_bbox_embed else None

        # Process class_embed if it exists and is multi-head
        if is_multi_head_class:
            original_class_embed_list = pruned_model.class_embed # Get from top-level model
            num_original_heads = len(original_class_embed_list)
            print(f"  Processing {num_original_heads} original class prediction heads.")
            # The final head corresponds to the output of the *last original* decoder layer
            final_head_original_idx = num_original_heads - 1

            num_kept_intermediate_heads = 0
            # Iterate through the indices of the decoder layers we *kept*
            for original_idx in dec_indices_to_keep:
                # If the kept decoder layer index corresponds to an *intermediate* head
                if original_idx < final_head_original_idx:
                    # Check if a head actually existed for this layer in the original model
                    if original_idx < num_original_heads:
                        # print(f"  Keeping original intermediate class head {original_idx} -> new head index {num_kept_intermediate_heads}")
                        new_class_embed_list.append(original_class_embed_list[original_idx])
                        num_kept_intermediate_heads += 1
                    else:
                        # This shouldn't happen if head count matched layer count
                        print(f"    Warning: Original class head index {original_idx} missing or out of bounds ({num_original_heads}). Skipping.")

            # Always append the *original final* prediction head
            if final_head_original_idx < num_original_heads:
                 print(f"  Appending original FINAL class head (index {final_head_original_idx}) as new head index {num_kept_intermediate_heads}.")
                 new_class_embed_list.append(original_class_embed_list[final_head_original_idx])
            else:
                 print(f"ERROR: Original final class head index {final_head_original_idx} seems out of bounds ({num_original_heads})!")

            pruned_model.class_embed = new_class_embed_list # Assign updated list to top-level model
            print(f"  Finished class heads. New count: {len(new_class_embed_list)}")

        elif has_class_embed: # Original class embed exists but is not a ModuleList (or has len 1)
             print("  Original class embed is single layer. Keeping it as is.")
             # No change needed, deepcopy already handled it.

        # Process bbox_embed if it exists and is multi-head (similar logic)
        if is_multi_head_bbox:
            original_bbox_embed_list = pruned_model.bbox_embed # Get from top-level model
            num_original_heads = len(original_bbox_embed_list)
            print(f"  Processing {num_original_heads} original bbox prediction heads.")
            final_head_original_idx = num_original_heads - 1

            num_kept_intermediate_heads = 0
            for original_idx in dec_indices_to_keep:
                 if original_idx < final_head_original_idx:
                     if original_idx < num_original_heads:
                         # print(f"  Keeping original intermediate bbox head {original_idx} -> new head index {num_kept_intermediate_heads}")
                         new_bbox_embed_list.append(original_bbox_embed_list[original_idx])
                         num_kept_intermediate_heads += 1
                     else:
                         print(f"    Warning: Original bbox head index {original_idx} missing or out of bounds ({num_original_heads}). Skipping.")

            # Always append the *original final* prediction head
            if final_head_original_idx < num_original_heads:
                 print(f"  Appending original FINAL bbox head (index {final_head_original_idx}) as new head index {num_kept_intermediate_heads}.")
                 new_bbox_embed_list.append(original_bbox_embed_list[final_head_original_idx])
            else:
                 print(f"ERROR: Original final bbox head index {final_head_original_idx} seems out of bounds ({num_original_heads})!")

            pruned_model.bbox_embed = new_bbox_embed_list # Assign updated list to top-level model
            print(f"  Finished bbox heads. New count: {len(new_bbox_embed_list)}")

        elif has_bbox_embed: # Original bbox embed exists but is single
             print("  Original bbox embed is single layer/module. Keeping it as is.")
             # No change needed

    elif has_class_embed: # Only a single head existed originally
         print("\nModel appears to have only a single final prediction head. No head pruning/re-indexing needed.")
    else:
         # No prediction heads found
         print("\nWarning: Model does not appear to have 'class_embed'. Skipping head processing.")


    print("--- Pruning Finished ---")
    print(f"Final layer counts: Encoder={pruned_model.num_encoder_layers}, Decoder={pruned_model.num_decoder_layers}")
    # Print final head counts for verification
    if hasattr(pruned_model, 'class_embed') and pruned_model.class_embed is not None:
        is_list = isinstance(pruned_model.class_embed, nn.ModuleList)
        head_len = len(pruned_model.class_embed) if is_list else 1 # Assume 1 if not list
        print(f"Final class_embed: {'ModuleList' if is_list else 'Single Module'}, Length/Count: {head_len}")
    if hasattr(pruned_model, 'bbox_embed') and pruned_model.bbox_embed is not None:
        is_list = isinstance(pruned_model.bbox_embed, nn.ModuleList)
        head_len = len(pruned_model.bbox_embed) if is_list else 1 # Assume 1 if not list
        print(f"Final bbox_embed: {'ModuleList' if is_list else 'Single Module'}, Length/Count: {head_len}")

    # Ensure the model configuration reflects the pruned layer counts (important for saving/reloading)
    if hasattr(pruned_model, 'config'):
        print("Updating model config with new layer counts...")
        pruned_model.config.encoder_layers = pruned_model.num_encoder_layers
        pruned_model.config.decoder_layers = pruned_model.num_decoder_layers
    else:
        print("Warning: Pruned model does not have a 'config' attribute to update layer counts.")


    return pruned_model

In [14]:
# =============================================================================
# Cell 5: Layer Pruning and Evaluation Loop (Using prune_model.py)
# =============================================================================
import torch
import torch.nn as nn
import copy # Keep copy, might be needed if prune_model doesn't deepcopy (it should)
import gc
import os
# Remove torch_pruning imports if no longer needed elsewhere
# import torch_pruning as tp
from thop import profile # Keep thop for GFLOPs
from tqdm.notebook import tqdm as tqdm_notebook
from tqdm import tqdm as tqdm_cli
import traceback
import time

# --- Make sure the pruning function is available ---


assert 'simplified_prune_detr' in globals(), "Run the cell defining simplified_prune_detr first!"
# --- End Pruning Function Check ---


print("\n--- Starting Layer Pruning & Evaluation Loop ---")

# Ensure baseline model and metrics exist
assert 'prepared_base_model' in locals() and prepared_base_model is not None, "Run Cell 2 first"
assert 'baseline_results' in locals(), "Run Cell 4 first to get baseline metrics"
assert 'TARGET_PRUNING_RATIOS' in locals(), "TARGET_PRUNING_RATIOS not defined in Cell 1"
assert 'image_processor' in locals() and image_processor is not None, "Image processor needed"
assert 'coco_gt' in locals(), "COCO GT object needed" # Allow None, but mAP will be skipped
assert 'eval_loader' in locals(), "Eval loader needed" # Allow None, but mAP will be skipped
assert 'output_dir' in locals(), "Output directory needed"
assert 'CONFIDENCE_THRESHOLD' in locals(), "Confidence threshold needed"
device = next(prepared_base_model.parameters()).device # Get device from model

# Use dummy input from baseline calculation if available, otherwise recreate
if 'dummy_input' not in locals() or dummy_input is None:
    print("Recreating dummy input for GFLOPs/timing...")
    bs = 1; img_h, img_w = 800, 1333 # Default/common size
    if hasattr(image_processor, 'size') and isinstance(image_processor.size, dict): # Simplified size logic
         size_dict = image_processor.size
         if 'shortest_edge' in size_dict: shortest = size_dict['shortest_edge']; max_size = getattr(image_processor, 'max_size', 1333); img_h = shortest; img_w = int(shortest * (1333/800)); img_w = min(img_w, max_size)
         elif 'height' in size_dict and 'width' in size_dict: img_h = size_dict['height']; img_w = size_dict['width']
         img_h = max(img_h, 32); img_w = max(img_w, 32)
    dummy_input = torch.randn(bs, 3, img_h, img_w, device=device)
    print(f"  Created dummy input shape: {dummy_input.shape} on {dummy_input.device}")
elif dummy_input.device != device:
    dummy_input = dummy_input.to(device)
    print(f"Moved existing dummy input to {dummy_input.device}")

pruning_results = {} # Dictionary to store metrics for each ratio

# --- Loop through each target pruning ratio ---
for ratio in TARGET_PRUNING_RATIOS:
    print(f"\n===== Processing Ratio: {ratio:.2f} (Layer Pruning) =====")
    pruning_results[ratio] = {} # Initialize dict for this ratio
    # Variables for this iteration scope
    model_pruned = None # <<< Renamed from model_pruned_copy
    coco_results_pruned = None
    coco_dt_pruned = None
    coco_eval_pruned = None

    try:
        # 1. Apply Layer Pruning using simplified_prune_detr
        #    This function should handle the deepcopy internally.
        print(f"  1. Applying simplified_prune_detr (ratio={ratio:.2f})...")
        pruning_start_time = time.time()
        # Pass the *original prepared base model* each time
        model_pruned = simplified_prune_detr(prepared_base_model, prune_ratio=ratio)
        pruning_end_time = time.time()
        model_pruned.to(device) # Ensure pruned model is on the correct device
        model_pruned.eval()
        print(f"     Layer pruning completed in {pruning_end_time - pruning_start_time:.2f} seconds.")
        print(f"     Pruned model moved to {next(model_pruned.parameters()).device}")

        # 2. Calculate Pruned Physical Metrics (Params, GFLOPs)
        print("  2. Calculating pruned physical metrics...")
        # Parameters
        pruned_params = sum(p.numel() for p in model_pruned.parameters() if p.requires_grad)
        pruning_results[ratio]['params'] = pruned_params
        pruning_results[ratio]['param_reduc%'] = (1 - pruned_params / baseline_results['params']) * 100 if baseline_results['params'] > 0 else 0
        print(f"     Pruned Params: {pruned_params:,} (Reduction: {pruning_results[ratio]['param_reduc%']:.2f}%)")

        # GFLOPs (using thop on the pruned model)
        pruning_results[ratio]['gflops'] = -1.0 # Initialize
        pruning_results[ratio]['gflop_reduc%'] = 0.0 # Initialize
        try:
            print("     Calculating GFLOPs using thop...")
            # Ensure dummy input matches pruned model device
            if dummy_input.device != model_pruned.device:
                 dummy_input = dummy_input.to(model_pruned.device)
            flops, params_thop = profile(model_pruned, inputs=(dummy_input,), verbose=False)
            pruned_gflops = flops / 1e9
            pruning_results[ratio]['gflops'] = pruned_gflops
            pruning_results[ratio]['gflop_reduc%'] = (1 - pruned_gflops / baseline_results['gflops']) * 100 if baseline_results['gflops'] > 0 else 0
            print(f"     Pruned GFLOPs: {pruned_gflops:.2f} (Reduction: {pruning_results[ratio]['gflop_reduc%']:.2f}%)")
            del flops, params_thop # Clean up thop variables
        except Exception as e_gflops_pruned:
            print(f"     Error calculating pruned GFLOPs: {e_gflops_pruned}")
            traceback.print_exc() # Print traceback for GFLOPs errors


        # 3. Calculate Pruned Performance Metrics (mAP, Inference Time)
        print("  3. Calculating pruned performance metrics...")
        pruning_results[ratio]['mAP'] = -1.0
        pruning_results[ratio]['mAP50'] = -1.0
        pruning_results[ratio]['avg_inference_ms'] = -1.0

        if coco_gt is None or eval_loader is None:
            print("     Skipping mAP/Inference Time (COCO GT or Eval Loader missing).")
        else:
            coco_results_pruned = [] # Re-initialize list for this ratio
            total_inference_time_ms_pruned = 0; processed_images_count_pruned = 0; inference_batches_pruned = 0
            model_pruned.eval() # Ensure eval mode
            with torch.no_grad():
                for batch_data in tqdm_notebook(eval_loader, desc=f"Evaluating Ratio {ratio:.2f}", leave=False):
                    if batch_data is None: continue
                    images, targets = batch_data

                    try:
                        # --- Batch Preprocessing ---
                        inputs = image_processor(images=images, return_tensors="pt").to(device)
                        original_sizes = [(t['height'], t['width']) for t in targets]
                        target_sizes = torch.tensor(original_sizes, device=device)
                        image_ids = [t['image_id'] for t in targets]

                        # --- Timed Inference ---
                        start_time = time.perf_counter()
                        outputs = model_pruned(**inputs) # Use the pruned model
                        end_time = time.perf_counter()
                        total_inference_time_ms_pruned += (end_time - start_time) * 1000
                        inference_batches_pruned += 1
                        # --- End Timed Inference ---

                        # --- Post-processing ---
                        results_list = image_processor.post_process_object_detection(
                            outputs,
                            target_sizes=target_sizes,
                            threshold=CONFIDENCE_THRESHOLD
                        )

                        # --- Format for COCO ---
                        for i, results in enumerate(results_list):
                            image_id = image_ids[i]
                            boxes = results["boxes"].cpu().tolist(); scores = results["scores"].cpu().tolist(); labels = results["labels"].cpu().tolist()
                            for box, score, label in zip(boxes, scores, labels):
                                x_min, y_min, x_max, y_max = box; coco_bbox = [x_min, y_min, x_max - x_min, y_max - y_min]
                                coco_results_pruned.append({
                                    "image_id": image_id,
                                    "category_id": label + 1, # Apply +1 correction for COCO IDs
                                    "bbox": coco_bbox,
                                    "score": score,
                                })
                            processed_images_count_pruned += 1
                        # --- Cleanup batch variables ---
                        del images, targets, inputs, outputs, results_list, original_sizes, target_sizes, image_ids
                        if 'results' in locals(): del results
                        if 'box' in locals(): del box, score, label, coco_bbox # Delete loop vars too
                        # --- End Cleanup ---
                    except Exception as e_infer_pruned:
                         print(f"\nError during pruned inference (Ratio {ratio:.2f}): {e_infer_pruned}")
                         continue # Skip batch on error

            print(f"     Processed {processed_images_count_pruned} images over {inference_batches_pruned} batches.")

            # Calculate Avg Inference Time
            if inference_batches_pruned > 0:
                 pruning_results[ratio]['avg_inference_ms'] = total_inference_time_ms_pruned / inference_batches_pruned
                 print(f"     Pruned Avg Batch Inference Time: {pruning_results[ratio]['avg_inference_ms']:.2f} ms")

            # Run COCOeval API
            if not coco_results_pruned:
                print("     No pruned evaluation results generated.")
            else:
                print("     Running COCO evaluation API for pruned model...")
                try:
                    coco_dt_pruned = coco_gt.loadRes(coco_results_pruned)
                    coco_eval_pruned = COCOeval(coco_gt, coco_dt_pruned, iouType='bbox')
                    coco_eval_pruned.evaluate(); coco_eval_pruned.accumulate(); coco_eval_pruned.summarize()
                    pruning_results[ratio]['mAP'] = coco_eval_pruned.stats[0]
                    pruning_results[ratio]['mAP50'] = coco_eval_pruned.stats[1]
                    print(f"     Pruned mAP: {pruning_results[ratio]['mAP']:.4f}, mAP50: {pruning_results[ratio]['mAP50']:.4f}")
                except Exception as e_coco_api_pruned:
                    print(f"     ERROR during pruned COCOeval: {e_coco_api_pruned}")
                    traceback.print_exc()
                finally: # Ensure cleanup even if COCO eval fails
                    if coco_dt_pruned is not None: del coco_dt_pruned
                    if coco_eval_pruned is not None: del coco_eval_pruned


        # 4. Save Pruned Model Structure (Optional but recommended)
        try:
            # NOTE: Saving using save_pretrained is preferred for HF models
            pruned_model_save_dir = os.path.join(output_dir, f"layer_pruned_ratio_{ratio:.2f}")
            print(f"  4. Saving layer-pruned structure to: {pruned_model_save_dir}")
            os.makedirs(pruned_model_save_dir, exist_ok=True)

            # --- Save using save_pretrained ---
            # This saves config, weights, processor config if available
            model_pruned.save_pretrained(pruned_model_save_dir)
            if image_processor:
                image_processor.save_pretrained(pruned_model_save_dir)
            pruning_results[ratio]['save_path'] = pruned_model_save_dir # Store path
            print(f"     Saved successfully using save_pretrained.")

        except Exception as e_save:
            print(f"     Error saving layer-pruned model for ratio {ratio:.2f}: {e_save}")
            pruning_results[ratio]['save_path'] = "Error Saving"


    # --- Error Handling for the whole iteration ---
    except Exception as e_ratio:
        print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
        print(f"!! ERROR processing ratio {ratio:.2f}: {e_ratio}")
        print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
        traceback.print_exc()
        pruning_results[ratio] = {'error': str(e_ratio), 'params': -1, 'gflops': -1, 'mAP': -1.0, 'mAP50': -1.0, 'avg_inference_ms': -1.0}


    # --- Explicit Cleanup within the loop ---
    finally:
        print(f"--- Cleaning up after ratio {ratio:.2f} ---")
        # Check memory BEFORE cleanup
        print(" >> Memory BEFORE cleanup:")
        try:
            print(torch.cuda.memory_summary(device=device, abbreviated=True))
        except Exception as mem_err:
            print(f"   Error getting memory summary: {mem_err}")

        # Delete objects created in this iteration
        if model_pruned is not None:
            print("   Deleting model_pruned...")
            del model_pruned
            model_pruned = None
        # --- Check before deleting COCO objects ---
        if 'coco_results_pruned' in locals() and coco_results_pruned is not None:
            print("   Deleting coco_results_pruned list...")
            del coco_results_pruned
        if 'coco_dt_pruned' in locals() and coco_dt_pruned is not None:
            print("   Deleting coco_dt_pruned object...")
            del coco_dt_pruned
        if 'coco_eval_pruned' in locals() and coco_eval_pruned is not None:
            print("   Deleting coco_eval_pruned object...")
            del coco_eval_pruned
        # --- End check ---
        # Delete other helper objects if they exist
        if 'ignored_layers' in locals() and ignored_layers is not None: del ignored_layers
        if 'importance' in locals() and importance is not None: del importance
        if 'nested_model' in locals() and nested_model is not None: del nested_model
        # Delete any lingering batch variables
        if 'images' in locals(): del images
        if 'targets' in locals(): del targets
        if 'inputs' in locals(): del inputs
        if 'outputs' in locals(): del outputs
        if 'results_list' in locals(): del results_list
        if 'batch_data' in locals(): del batch_data

        # Force Python garbage collection
        print("   Running gc.collect()...")
        gc.collect()

        # Ask PyTorch to release cached memory
        if torch.cuda.is_available():
            print("   Running torch.cuda.empty_cache()...")
            torch.cuda.empty_cache()

        # Check memory AFTER cleanup
        print(" >> Memory AFTER cleanup:")
        try:
            print(torch.cuda.memory_summary(device=device, abbreviated=True))
        except Exception as mem_err:
            print(f"   Error getting memory summary: {mem_err}")
        print(f"--- End Cleanup for ratio {ratio:.2f} ---")


print("\n===== Layer Pruning & Evaluation Loop Finished =====")
# Store results globally if needed by next cells
globals()['pruning_results'] = pruning_results
globals()['baseline_results'] = baseline_results # Make sure baseline is also global
if 'dummy_input' in locals(): globals()['dummy_input'] = dummy_input # Keep dummy input if needed


--- Starting Layer Pruning & Evaluation Loop ---

===== Processing Ratio: 0.10 (Layer Pruning) =====
  1. Applying simplified_prune_detr (ratio=0.10)...

--- Starting Simplified Layer Pruning (Ratio: 0.10) ---
Original encoder layers: 6
Original decoder layers: 6
Targeting removal of 1 encoder layers.
Targeting removal of 1 decoder layers.
Keeping encoder layers at original indices: [0, 1, 2, 4, 5]
Keeping decoder layers at original indices: [0, 2, 3, 4, 5]

Rebuilding prediction heads ModuleList for pruned decoder...
  Processing 6 original class prediction heads.
  Appending original FINAL class head (index 5) as new head index 4.
  Finished class heads. New count: 5
  Processing 6 original bbox prediction heads.
  Appending original FINAL bbox head (index 5) as new head index 4.
  Finished bbox heads. New count: 5
--- Pruning Finished ---
Final layer counts: Encoder=5, Decoder=5
Final class_embed: ModuleList, Length/Count: 5
Final bbox_embed: ModuleList, Length/Count: 5
Updating mo

Evaluating Ratio 0.10:   0%|          | 0/200 [00:00<?, ?it/s]

     Processed 200 images over 200 batches.
     Pruned Avg Batch Inference Time: 169.80 ms
     Running COCO evaluation API for pruned model...
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.58s).
Accumulating evaluation results...
DONE (t=0.09s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.098
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.266
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.042
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.046
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.104
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.163
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.072
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.250
 Av

Evaluating Ratio 0.20:   0%|          | 0/200 [00:00<?, ?it/s]

Exception ignored in: <function tqdm.__del__ at 0x7d7e355cfd80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/tqdm/std.py", line 1147, in __del__
    def __del__(self):

KeyboardInterrupt: 


--- Cleaning up after ratio 0.20 ---
 >> Memory BEFORE cleanup:
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      | 325485 KiB |   1689 MiB |   5435 GiB |   5435 GiB |
|---------------------------------------------------------------------------|
| Active memory         | 325485 KiB |   1689 MiB |   5435 GiB |   5435 GiB |
|---------------------------------------------------------------------------|
| Requested memory      | 320629 KiB |   1679 MiB |   5398 GiB |   5398 GiB |


KeyboardInterrupt: 

In [ ]:
# =============================================================================
# Cell 6: Final Summary and Save Results (MODIFIED)
# =============================================================================
import json       # <-- Added import
import pandas as pd
import os
import traceback  # <-- Added import for robust error handling

print("\n--- Final Summary and Saving Results ---")

# Ensure results dictionaries exist
assert 'baseline_results' in locals(), "Run Cell 4 first for baseline metrics."
assert 'pruning_results' in locals(), "Run Cell 5/5.5 first for pruning loop results."
assert 'output_dir' in locals(), "Output directory must be defined in Cell 1."

# --- Define Output Filenames ---
json_filename = "pruning_results_summary.json"
csv_filename = "pruning_results_summary.csv"
json_filepath = os.path.join(output_dir, json_filename)
csv_filepath = os.path.join(output_dir, csv_filename)

# --- Combine Results for JSON Saving ---
# Create a single dictionary holding both baseline and pruned results
all_results_to_save = {
    "baseline": baseline_results,
    "pruning_ratios": pruning_results # This already contains results keyed by ratio
}

# --- Save to JSON ---
print(f"\nSaving detailed results to JSON: {json_filepath}")
try:
    with open(json_filepath, 'w') as f:
        # Use indent for readability
        json.dump(all_results_to_save, f, indent=4, sort_keys=True)
    print("  JSON results saved successfully.")
except Exception as e_json:
    print(f"  ERROR saving results to JSON: {e_json}")
    traceback.print_exc()

# --- Prepare Data for CSV and Print Summary ---
print("\n--- Baseline Model Metrics (for reference) ---")
print(f"Parameters: {baseline_results.get('params', 'N/A'):,}")
print(f"GFLOPs: {baseline_results.get('gflops', -1.0):.2f}")
print(f"mAP: {baseline_results.get('mAP', -1.0):.4f}")
print(f"mAP50: {baseline_results.get('mAP50', -1.0):.4f}")
print(f"Avg Inference Time (ms/batch): {baseline_results.get('avg_inference_ms', -1.0):.2f}")
print("-" * 90)

print("\n--- Pruned Model Metrics Summary ---")
data_for_df = []

# Add baseline as the first row for comparison in the CSV/DataFrame
data_for_df.append({
    'Ratio': "Baseline",
    'Params': f"{baseline_results.get('params', 0):,}",
    'Param Reduc %': "0.00",
    'GFLOPs': f"{baseline_results.get('gflops', -1.0):.2f}",
    'GFLOP Reduc %': "0.00",
    'mAP': f"{baseline_results.get('mAP', -1.0):.4f}" if baseline_results.get('mAP', -1.0) >= 0 else "N/A",
    'mAP50': f"{baseline_results.get('mAP50', -1.0):.4f}" if baseline_results.get('mAP50', -1.0) >= 0 else "N/A",
    'Avg Inf Time (ms)': f"{baseline_results.get('avg_inference_ms', -1.0):.2f}" if baseline_results.get('avg_inference_ms', -1.0) >= 0 else "N/A",
    'Save Path': "N/A (Baseline)"
})

# Add results for each pruning ratio
sorted_ratios = sorted(pruning_results.keys())
for ratio in sorted_ratios:
    res = pruning_results[ratio]
    if 'error' in res:
         data_for_df.append({
            'Ratio': f"{ratio:.2f}",
            'Params': "ERROR", 'Param Reduc %': "ERROR",
            'GFLOPs': "ERROR", 'GFLOP Reduc %': "ERROR",
            'mAP': "ERROR", 'mAP50': "ERROR",
            'Avg Inf Time (ms)': "ERROR",
            'Save Path': res.get('save_path', 'ERROR')
         })
         # Optionally print error during summary generation
         # print(f"\nERROR details for ratio {ratio:.2f}: {res['error']}")
    elif res.get('params', -1) == -1 and res.get('gflops', -1.0) == -1.0:
        # Handle cases where loop might have skipped without explicit error key
         data_for_df.append({
            'Ratio': f"{ratio:.2f}",
            'Params': "SKIPPED", 'Param Reduc %': "SKIPPED",
            'GFLOPs': "SKIPPED", 'GFLOP Reduc %': "SKIPPED",
            'mAP': "SKIPPED", 'mAP50': "SKIPPED",
            'Avg Inf Time (ms)': "SKIPPED",
            'Save Path': res.get('save_path', 'SKIPPED')
         })
    else:
        # Format valid results
        data_for_df.append({
            'Ratio': f"{ratio:.2f}",
            'Params': f"{res.get('params', 0):,}",
            'Param Reduc %': f"{res.get('param_reduc%', 0.0):.2f}",
            'GFLOPs': f"{res.get('gflops', -1.0):.2f}" if res.get('gflops', -1.0) >= 0 else "Error",
            'GFLOP Reduc %': f"{res.get('gflop_reduc%', 0.0):.2f}" if res.get('gflops', -1.0) >= 0 else "Error",
            'mAP': f"{res.get('mAP', -1.0):.4f}" if res.get('mAP', -1.0) >= 0 else "N/A",
            'mAP50': f"{res.get('mAP50', -1.0):.4f}" if res.get('mAP50', -1.0) >= 0 else "N/A",
            'Avg Inf Time (ms)': f"{res.get('avg_inference_ms', -1.0):.2f}" if res.get('avg_inference_ms', -1.0) >= 0 else "N/A",
            'Save Path': res.get('save_path', 'N/A')
        })

# --- Create DataFrame ---
df = pd.DataFrame(data_for_df)
# Set Ratio as index AFTER creating DataFrame for better control when saving CSV
df.set_index('Ratio', inplace=True)

# --- Save to CSV ---
print(f"\nSaving results summary table to CSV: {csv_filepath}")
try:
    # index=True saves the 'Ratio' column which is now the index
    df.to_csv(csv_filepath, index=True)
    print("  CSV results saved successfully.")
except Exception as e_csv:
    print(f"  ERROR saving results to CSV: {e_csv}")
    traceback.print_exc()

# --- Print Summary Table To Output ---
print("\n--- Summary Table ---")
if not df.empty:
    # Display relevant columns - adjust columns if needed
    display_columns = ['Params', 'Param Reduc %', 'GFLOPs', 'GFLOP Reduc %', 'mAP', 'mAP50', 'Avg Inf Time (ms)']
    # Filter out columns that might not exist if all runs failed, etc.
    display_columns = [col for col in display_columns if col in df.columns]
    if display_columns:
         print(df[display_columns])
    else:
         print("No valid metric columns found to display in table.")
else:
    print("No pruning results to display.")

# --- Print Saved Model Locations ---
print("\n--- Saved Model Locations ---")
if not df.empty and 'Save Path' in df.columns:
    # Iterate through the DataFrame to print paths cleanly
    for ratio_idx, row in df.iterrows():
         print(f"Ratio {ratio_idx}: {row['Save Path']}")
else:
     print("No models were saved or results DataFrame is empty.")

print("\n--- End of Summary ---")


--- Final Summary and Saving Results ---

Saving detailed results to JSON: /content/drive/MyDrive/kitti_torch_pruning_results_v2/pruning_results_summary.json
  JSON results saved successfully.

--- Baseline Model Metrics (for reference) ---
Parameters: 39,877,769
GFLOPs: 204.87
mAP: 0.1168
mAP50: 0.2639
Avg Inference Time (ms/batch): 199.61
------------------------------------------------------------------------------------------

--- Pruned Model Metrics Summary ---

Saving results summary table to CSV: /content/drive/MyDrive/kitti_torch_pruning_results_v2/pruning_results_summary.csv
  CSV results saved successfully.

--- Summary Table ---
              Params Param Reduc %  GFLOPs GFLOP Reduc %     mAP   mAP50  \
Ratio                                                                      
Baseline  39,877,769          0.00  204.87          0.00  0.1168  0.2639   
0.10      36,535,561          8.38   Error         Error  0.0915  0.2250   
0.20      33,827,049         15.17   Error    